# runML Universal — Case Study

Trains a universal LSTM on 10 symbols per year, then evaluates across every
**3-month**, **6-month**, and **1-year** sub-period.

For each (symbol, period) a figure with two subplots is saved:
- **(a)** actual close price vs predicted probability
- **(b)** bar plot: +1 = correct prediction, −1 = wrong prediction

In [13]:
import os
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import torch
from datetime import datetime
from pathlib import Path
from torch.utils.data import DataLoader
from sklearn.preprocessing import StandardScaler

from sentiment.log import setup_logging
from sentiment.sources.alpaca import AlpacaSource
from sentiment.sources.cache import MarketDataCache
from sentiment.sources.fundamental import FundamentalSource, FundamentalCache
from sentiment.features.technical import TechnicalFactors
from sentiment.features.dataloader import (
    FusedDataset, FusedStockDataset, build_dataset, make_loaders
)
from sentiment.model.lstm import SentimentLSTM
from sentiment.model.transformer import SentimentTransformer
from sentiment.model.train import train_model, bootstrap_evaluate, collect_predictions

setup_logging()

## Configuration

In [14]:
CASE_STUDY_YEARS = [2018, 2019, 2020]
LOAD_YEARS       = [2017, 2018, 2019, 2020]   # 2017 provides warmup for 2018 Q1
WARMUP_DAYS      = 150                         # calendar days to prepend before each period
WINDOW     = 64
BATCH_SIZE = 32
N_EPOCHS   = 100
MODEL_TYPE = "lstm"
DEVICE     = "cuda" if torch.cuda.is_available() else "cpu"

symbols = [
    "AAPL", "MSFT", "NVDA", "AMZN", "GOOGL",
    "META", "TSLA", "AVGO", "PEP", "COST",
]

PLOTS_DIR = Path("case_study_plots_with_news")
for sub in ("3months", "6months", "1year"):
    (PLOTS_DIR / sub).mkdir(parents=True, exist_ok=True)

PERIOD_SUBFOLDER = {"3m": "3months", "6m": "6months", "1y": "1year"}

print(f"Plots dir : {PLOTS_DIR.resolve()}")
print(f"Device    : {DEVICE}")

Plots dir : C:\TFA\ProjectCode\quant-sentiment-score_mdl_260326\notebooks\case_study_plots_with_news
Device    : cpu


## Helper Functions

In [15]:
def is_daily(df):
    """Return True when median gap between rows is >= 20 hours."""
    if len(df) < 2:
        return False
    gaps = df.index.to_series().diff().dropna()
    return gaps.median() >= pd.Timedelta("20h")


def generate_periods(year):
    """Return {period_type: [(start, end), ...]} for 3m, 6m, 1y."""
    return {
        "1y": [
            (datetime(year, 1, 1), datetime(year, 12, 31)),
        ],
        "6m": [
            (datetime(year, 1, 1), datetime(year, 6, 30)),
            (datetime(year, 7, 1), datetime(year, 12, 31)),
        ],
        "3m": [
            (datetime(year, 1,  1), datetime(year, 3, 31)),
            (datetime(year, 4,  1), datetime(year, 6, 30)),
            (datetime(year, 7,  1), datetime(year, 9, 30)),
            (datetime(year, 10, 1), datetime(year, 12, 31)),
        ],
    }


def make_eval_loader(ds, tech_scaler, fund_scaler, batch_size=32):
    """Build a non-shuffling DataLoader for a single symbol dataset."""
    X_tech = ds["X_tech"].copy()
    n, w, f = X_tech.shape
    X_tech = (
        tech_scaler.transform(X_tech.reshape(-1, f))
        .reshape(n, w, f)
        .astype(np.float32)
    )
    X_fund = ds["X_fundamental"].copy()
    if fund_scaler is not None:
        X_fund = fund_scaler.transform(X_fund).astype(np.float32)

    dataset = FusedStockDataset(
        X_tech, ds["X_sentiment"], X_fund, ds["X_sentiment_probs"], ds["y"]
    )
    return DataLoader(dataset, batch_size=batch_size, shuffle=False)

## Load Market Data

In [16]:
dfs_all = {}   # symbol -> concatenated DataFrame across all LOAD_YEARS

def strip_tz(df):
    """Remove timezone from DatetimeIndex so comparisons with naive timestamps work."""
    if df.index.tz is not None:
        df = df.copy()
        df.index = df.index.tz_convert("UTC").tz_localize(None)
    return df

for year in LOAD_YEARS:
    print(f"\n=== Loading {year} ===")
    start  = datetime(year, 1, 1)
    end    = datetime(year, 12, 31)
    cache  = MarketDataCache()
    alpaca = AlpacaSource()
    for symbol in symbols:
        df = None
        try:
            df = cache.load(symbol, year)
            if df.empty or not is_daily(df):
                df = None
            else:
                print(f"  {symbol}: loaded {len(df)} cached bars")
        except Exception:
            pass

        if df is None:
            try:
                df = alpaca.fetch_bars(symbol, start, end, timeframe="1Day")
                cache.store(symbol, year, df)
                print(f"  {symbol}: fetched {len(df)} bars")
            except Exception as e:
                print(f"  {symbol} {year}: fetch failed — {e}, skipping")
                continue

        df = strip_tz(df)

        if symbol not in dfs_all:
            dfs_all[symbol] = df
        else:
            dfs_all[symbol] = pd.concat([dfs_all[symbol], df]).sort_index()

print(f"\nLoaded {len(dfs_all)} symbols across years {LOAD_YEARS}")


=== Loading 2017 ===
  AAPL 2017: fetch failed — AlpacaSource.fetch_bars() got multiple values for argument 'timeframe', skipping
  MSFT 2017: fetch failed — AlpacaSource.fetch_bars() got multiple values for argument 'timeframe', skipping
  NVDA 2017: fetch failed — AlpacaSource.fetch_bars() got multiple values for argument 'timeframe', skipping
  AMZN 2017: fetch failed — AlpacaSource.fetch_bars() got multiple values for argument 'timeframe', skipping
  GOOGL 2017: fetch failed — AlpacaSource.fetch_bars() got multiple values for argument 'timeframe', skipping
  META 2017: fetch failed — AlpacaSource.fetch_bars() got multiple values for argument 'timeframe', skipping
  TSLA 2017: fetch failed — AlpacaSource.fetch_bars() got multiple values for argument 'timeframe', skipping
  AVGO 2017: fetch failed — AlpacaSource.fetch_bars() got multiple values for argument 'timeframe', skipping
  PEP 2017: fetch failed — AlpacaSource.fetch_bars() got multiple values for argument 'timeframe', skippi

## Load Fundamentals

In [17]:
fund_source = FundamentalSource()
fund_cache  = FundamentalCache()
fund_dfs    = {}

for symbol in symbols:
    fund_df = fund_cache.load_df(symbol)
    if fund_df is None:
        print(f"  {symbol}: fetching fundamentals...")
        fund_data = fund_source.fetch(symbol)
        fund_cache.store(symbol, fund_data)
        fund_df = fund_cache.load_df(symbol)
    else:
        print(f"  {symbol}: loaded cached fundamentals")
    fund_dfs[symbol] = fund_df

  AAPL: loaded cached fundamentals
  MSFT: loaded cached fundamentals
  NVDA: loaded cached fundamentals
  AMZN: loaded cached fundamentals
  GOOGL: loaded cached fundamentals
  META: loaded cached fundamentals
  TSLA: loaded cached fundamentals
  AVGO: loaded cached fundamentals
  PEP: loaded cached fundamentals
  COST: loaded cached fundamentals


## Load Sentiment

In [18]:
import glob as _glob, importlib.util as _ilu
import numpy as _np

# ------------------------------------------------------------------
# Load SentimentCache + SentimentEncoder directly from project files
# (Jupyter kernel may resolve the wrong 'sentiment' package)
# ------------------------------------------------------------------
_PROJECT_ROOT = Path.cwd().parent   # notebooks/ -> project root

def _load_mod(name, rel_path):
    spec = _ilu.spec_from_file_location(name, _PROJECT_ROOT / rel_path)
    mod  = _ilu.module_from_spec(spec)
    spec.loader.exec_module(mod)
    return mod

SentimentCache   = _load_mod("_sc_cache",   "sentiment/embeddings/cache.py").SentimentCache
SentimentEncoder = _load_mod("_sc_enc",     "sentiment/embeddings/encoder.py").SentimentEncoder

def _aggregate_daily(df):
    """Mirror of pipeline.aggregate_daily — collapse per-article rows to daily means."""
    agg = (
        df.groupby(["ticker", "date"])
        .agg(sentiment_score=("label", "mean"), n_articles=("label", "count"))
        .reset_index()
    )
    for col in [c for c in ["embedding", "sentiment_probs"] if c in df.columns]:
        arr    = _np.stack(df[col].values)                    # (M, D)
        arr_df = pd.DataFrame(arr, index=df.index)
        means  = arr_df.groupby([df["ticker"], df["date"]]).mean().reset_index()
        means[col] = list(means.iloc[:, 2:].values.astype(_np.float32))
        means  = means[["ticker", "date", col]]
        agg    = agg.merge(means, on=["ticker", "date"])
    return agg

# ------------------------------------------------------------------
# News articles — same source as CompletePipeline.ipynb
# (ArticleRepository reads from quant-sentiment-score/data/news/)
# ------------------------------------------------------------------
NEWS_DIR = Path("C:/TFA/ProjectCode/quant-sentiment-score/data/news")

sentiment_cache = SentimentCache()   # stores to <project>/data/sentiment/<SYM>.parquet

# ------------------------------------------------------------------
# 1. Compute FinBERT embeddings for symbols not yet cached
# ------------------------------------------------------------------
symbols_need = [s for s in symbols if not sentiment_cache.exists(s)]

if symbols_need:
    news_files = sorted(_glob.glob(str(NEWS_DIR / "*.parquet")))
    if not news_files:
        print(f"WARNING: no news files found at {NEWS_DIR}")
    else:
        print(f"Computing embeddings for: {symbols_need}")
        print(f"  news files : {len(news_files)}  ({news_files[0][-14:]} … {news_files[-1][-14:]})")

        # Collect articles per symbol (same filter as CompletePipeline)
        articles_by_sym = {s: [] for s in symbols_need}
        for fpath in news_files:
            df_news = pd.read_parquet(fpath)
            for sym in symbols_need:
                mask = df_news["tickers"].apply(
                    lambda x: sym in list(x) if x is not None else False
                )
                sub = df_news[mask].copy()
                if not sub.empty:
                    sub["ticker"] = sym
                    articles_by_sym[sym].append(sub)

        device = "cuda" if torch.cuda.is_available() else "cpu"
        encoder = SentimentEncoder(device=device)
        print(f"  FinBERT loaded on {device}")

        for sym in symbols_need:
            frames = articles_by_sym[sym]
            if not frames:
                print(f"  {sym}: no articles in news files — zero vectors")
                continue

            df_sym = pd.concat(frames, ignore_index=True)
            print(f"  {sym}: encoding {len(df_sym)} articles …", end="", flush=True)

            rows = []
            for _, art in df_sym.iterrows():
                title = str(art.get("title") or "").strip()
                body  = str(art.get("text")  or "").strip()
                # Feed title + full body; FinBERT truncates at 512 tokens
                # (equivalent to CompletePipeline with summarizer_model=None)
                text  = f"{title} {body}".strip()
                if not text:
                    continue
                try:
                    label, emb, probs = encoder.encode(text)
                    rows.append({
                        "ticker":          sym,
                        "date":            art["publish_date"],
                        "label":           label,
                        "embedding":       emb,
                        "sentiment_probs": probs,
                    })
                except Exception:
                    pass

            if rows:
                daily_df = _aggregate_daily(pd.DataFrame(rows))
                sentiment_cache.store(sym, daily_df)
                print(f" done — {len(daily_df)} days cached")
            else:
                print(" no valid articles encoded")

# ------------------------------------------------------------------
# 2. Load from cache; fall back to None (→ zero vectors in build_dataset)
# ------------------------------------------------------------------
sentiment_dfs: dict = {}
for symbol in symbols:
    if sentiment_cache.exists(symbol):
        sentiment_dfs[symbol] = sentiment_cache.load(symbol)
        print(f"{symbol}: {len(sentiment_dfs[symbol])} days from sentiment cache")
    else:
        sentiment_dfs[symbol] = None
        print(f"{symbol}: no data — zero vectors")

Computing embeddings for: ['AAPL', 'MSFT', 'NVDA', 'AMZN', 'GOOGL', 'META', 'TSLA', 'AVGO', 'PEP', 'COST']
  news files : 30  (018-01.parquet … 020-06.parquet)


04:10:00 INFO     httpx  HTTP Request: HEAD https://huggingface.co/ProsusAI/finbert/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
04:10:01 INFO     httpx  HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/ProsusAI/finbert/4556d13015211d73dccd3fdd39d39232506f3e43/config.json "HTTP/1.1 200 OK"
04:10:01 INFO     httpx  HTTP Request: HEAD https://huggingface.co/ProsusAI/finbert/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
04:10:02 INFO     httpx  HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/ProsusAI/finbert/4556d13015211d73dccd3fdd39d39232506f3e43/tokenizer_config.json "HTTP/1.1 200 OK"
04:10:02 INFO     httpx  HTTP Request: GET https://huggingface.co/api/models/ProsusAI/finbert/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
04:10:03 INFO     httpx  HTTP Request: GET https://huggingface.co/api/models/ProsusAI/finbert/tree/main?recursive=true&expand=false "HTTP/1.1 200 OK"


  FinBERT loaded on cpu
  AAPL: encoding 463 articles …

04:10:06 INFO     httpx  HTTP Request: GET https://huggingface.co/api/models/ProsusAI/finbert/commits/main "HTTP/1.1 200 OK"
04:10:06 INFO     httpx  HTTP Request: GET https://huggingface.co/api/models/ProsusAI/finbert/discussions?p=0 "HTTP/1.1 200 OK"
04:10:07 INFO     httpx  HTTP Request: GET https://huggingface.co/api/models/ProsusAI/finbert/commits/refs%2Fpr%2F29 "HTTP/1.1 200 OK"
04:10:07 INFO     httpx  HTTP Request: HEAD https://huggingface.co/ProsusAI/finbert/resolve/refs%2Fpr%2F29/model.safetensors.index.json "HTTP/1.1 404 Not Found"
04:10:08 INFO     httpx  HTTP Request: HEAD https://huggingface.co/ProsusAI/finbert/resolve/refs%2Fpr%2F29/model.safetensors "HTTP/1.1 302 Found"


 done — 79 days cached
  MSFT: no articles in news files — zero vectors
  NVDA: encoding 1333 articles … done — 475 days cached
  AMZN: encoding 268 articles … done — 35 days cached
  GOOGL: encoding 1742 articles … done — 473 days cached
  META: no articles in news files — zero vectors
  TSLA: encoding 1865 articles … done — 283 days cached
  AVGO: encoding 2488 articles … done — 659 days cached
  PEP: encoding 459 articles … done — 211 days cached
  COST: no articles in news files — zero vectors
AAPL: 79 days from sentiment cache
MSFT: no data — zero vectors
NVDA: 475 days from sentiment cache
AMZN: 35 days from sentiment cache
GOOGL: 473 days from sentiment cache
META: no data — zero vectors
TSLA: 283 days from sentiment cache
AVGO: 659 days from sentiment cache
PEP: 211 days from sentiment cache
COST: no data — zero vectors


## Case Study Loop — Train per Year, Collect Predictions

In [19]:
# all_results[(period_type, start_str, end_str, symbol)] = test-only DataFrame
all_results = {}
technical   = TechnicalFactors()

# Flatten all periods across all case study years
all_periods = [
    (ptype, start, end)
    for year in CASE_STUDY_YEARS
    for ptype, periods in generate_periods(year).items()
    for start, end in periods
]

for period_type, start, end in all_periods:
    start_str    = start.strftime("%Y%m%d")
    end_str      = end.strftime("%Y%m%d")
    period_label = f"{start_str}-{end_str}"
    print(f"\n{'='*55}")
    print(f"  {period_type}: {period_label}")
    print(f"{'='*55}")

    # ---- build per-symbol datasets filtered to this period ----
    sym_filtered = {}
    warmup_start = pd.Timestamp(start) - pd.Timedelta(days=WARMUP_DAYS)

    for symbol in symbols:
        if symbol not in dfs_all:
            continue
        df_slice = dfs_all[symbol].loc[
            (dfs_all[symbol].index >= warmup_start) &
            (dfs_all[symbol].index <= pd.Timestamp(end))
        ]
        if len(df_slice) < 126:
            continue
        try:
            ds = build_dataset(
                df_slice, technical,
                sentiment_df=None, ticker=symbol,
                window=WINDOW,
                fundamental_df=fund_dfs.get(symbol),
                include_momentum_slope=True,
            )
        except RuntimeError as e:
            print(f"  {symbol}: skipped — {e}")
            continue

        # Keep only windows whose end date falls within [start, end]
        win_dates = pd.to_datetime(ds["dates"])
        mask = (win_dates >= pd.Timestamp(start)) & (win_dates <= pd.Timestamp(end))
        idx  = np.where(mask)[0]
        if len(idx) < 10:
            continue

        sym_filtered[symbol] = {
            "X_tech":            ds["X_tech"][idx],
            "X_sentiment":       ds["X_sentiment"][idx],
            "X_sentiment_probs": ds["X_sentiment_probs"][idx],
            "X_fundamental":     ds["X_fundamental"][idx],
            "y":                 ds["y"][idx],
            "dates":             win_dates[idx].values,
        }
        print(f"  {symbol}: {len(idx)} windows")

    if not sym_filtered:
        print("  No valid symbols, skipping period.")
        continue

    # ---- pool + sort by date ---------------------------------
    ds_list      = list(sym_filtered.values())
    pooled_tech  = np.concatenate([d["X_tech"]            for d in ds_list])
    pooled_sent  = np.concatenate([d["X_sentiment"]       for d in ds_list])
    pooled_sprob = np.concatenate([d["X_sentiment_probs"] for d in ds_list])
    pooled_fund  = np.concatenate([d["X_fundamental"]     for d in ds_list])
    pooled_y     = np.concatenate([d["y"]                 for d in ds_list])
    pooled_dates = np.concatenate([d["dates"]             for d in ds_list])

    sort_idx     = np.argsort(pooled_dates, kind="stable")
    pooled_tech  = pooled_tech[sort_idx];  pooled_sent  = pooled_sent[sort_idx]
    pooled_sprob = pooled_sprob[sort_idx]; pooled_fund  = pooled_fund[sort_idx]
    pooled_y     = pooled_y[sort_idx];     pooled_dates = pooled_dates[sort_idx]

    N             = len(pooled_y)
    test_start_i  = int(N * 0.8)
    val_start_i   = int(test_start_i * 0.9)

    if N - test_start_i < 5:
        print(f"  Too few test samples ({N - test_start_i}), skipping.")
        continue

    # ---- scale -----------------------------------------------
    n_all, w, f = pooled_tech.shape
    tech_scaler  = StandardScaler()
    tech_scaler.fit(pooled_tech[:val_start_i].reshape(-1, f))
    X_tech_sc = tech_scaler.transform(pooled_tech.reshape(-1, f)).reshape(n_all, w, f).astype(np.float32)

    fund_scaler  = None
    X_fund_sc    = pooled_fund.copy()
    if pooled_fund.shape[1] > 0:
        fund_scaler = StandardScaler()
        fund_scaler.fit(pooled_fund[:val_start_i])
        X_fund_sc = fund_scaler.transform(pooled_fund).astype(np.float32)

    # ---- data loaders ----------------------------------------
    def make_loader(sl, shuffle=False):
        obj = FusedStockDataset(
            X_tech_sc[sl], pooled_sent[sl], X_fund_sc[sl],
            pooled_sprob[sl], pooled_y[sl]
        )
        return DataLoader(obj, batch_size=BATCH_SIZE, shuffle=shuffle)

    train_loader = make_loader(slice(0, val_start_i), shuffle=True)
    val_loader   = make_loader(slice(val_start_i, test_start_i))

    # ---- model + train ---------------------------------------
    n_factors = pooled_tech.shape[2]
    n_fund    = pooled_fund.shape[1]
    n_sprob   = pooled_sprob.shape[-1]
    sent_dim  = pooled_sent.shape[-1]

    model = SentimentLSTM(
        n_factors=n_factors, sentiment_dim=sent_dim,
        hidden_size=32, num_layers=2, dropout=0.2,
        n_fundamentals=n_fund, n_sentiment_probs=n_sprob,
    )
    history = train_model(
        model, train_loader, val_loader,
        n_epochs=N_EPOCHS, lr=1e-3, patience=15,
        device=DEVICE, seed=42,
    )
    print(f"  Best epoch {history['best_epoch']}, val AUC={history['best_val_auc']:.4f} | test windows={N - test_start_i}")

    # ---- per-symbol test inference ---------------------------
    # Test date range comes from the globally sorted pooled data
    test_date_min = pd.Timestamp(pooled_dates[test_start_i])
    test_date_max = pd.Timestamp(pooled_dates[-1])

    for symbol, ds_f in sym_filtered.items():
        sym_dates = pd.to_datetime(ds_f["dates"])
        test_mask = (sym_dates >= test_date_min) & (sym_dates <= test_date_max)
        t_idx     = np.where(test_mask)[0]
        if len(t_idx) == 0:
            continue

        # Apply fitted scalers to this symbol's test windows
        Xt = ds_f["X_tech"][t_idx].copy()
        ns, ws, fs = Xt.shape
        Xt = tech_scaler.transform(Xt.reshape(-1, fs)).reshape(ns, ws, fs).astype(np.float32)
        Xf = ds_f["X_fundamental"][t_idx].copy()
        if fund_scaler is not None:
            Xf = fund_scaler.transform(Xf).astype(np.float32)

        sym_ds  = FusedStockDataset(Xt, ds_f["X_sentiment"][t_idx],
                                    Xf, ds_f["X_sentiment_probs"][t_idx],
                                    ds_f["y"][t_idx])
        sym_ldr = DataLoader(sym_ds, batch_size=BATCH_SIZE, shuffle=False)
        probs, actuals = collect_predictions(model, sym_ldr, device=DEVICE)

        dates_t      = sym_dates[t_idx]
        close_prices = dfs_all[symbol]["close"].reindex(dates_t).values
        pred_labels  = (probs > 0.5).astype(int)
        correct      = np.where(pred_labels == actuals.astype(int), 1, -1)

        all_results[(period_type, start_str, end_str, symbol)] = pd.DataFrame({
            "date":         dates_t,
            "close":        close_prices,
            "pred_prob":    probs,
            "actual_label": actuals.astype(int),
            "pred_label":   pred_labels,
            "correct":      correct,
        })

print("\nCase study loop done.")

04:19:31 INFO     sentiment.features.dataloader  Built dataset: 126 windows, 16 tech factors, 10 fundamental factors (incl. momentum slope), 0 sentiment prob features, window=64
04:19:31 INFO     sentiment.features.dataloader  Built dataset: 126 windows, 16 tech factors, 10 fundamental factors (incl. momentum slope), 0 sentiment prob features, window=64
04:19:31 INFO     sentiment.features.dataloader  Built dataset: 126 windows, 16 tech factors, 10 fundamental factors (incl. momentum slope), 0 sentiment prob features, window=64



  1y: 20180101-20181231
  AAPL: 126 windows
  MSFT: 126 windows
  NVDA: 126 windows


04:19:31 INFO     sentiment.features.dataloader  Built dataset: 126 windows, 16 tech factors, 10 fundamental factors (incl. momentum slope), 0 sentiment prob features, window=64
04:19:31 INFO     sentiment.features.dataloader  Built dataset: 126 windows, 16 tech factors, 10 fundamental factors (incl. momentum slope), 0 sentiment prob features, window=64
04:19:31 INFO     sentiment.features.dataloader  Built dataset: 126 windows, 16 tech factors, 10 fundamental factors (incl. momentum slope), 0 sentiment prob features, window=64
04:19:31 INFO     sentiment.features.dataloader  Built dataset: 126 windows, 16 tech factors, 10 fundamental factors (incl. momentum slope), 0 sentiment prob features, window=64


  AMZN: 126 windows
  GOOGL: 126 windows
  META: 126 windows
  TSLA: 126 windows


04:19:31 INFO     sentiment.features.dataloader  Built dataset: 126 windows, 16 tech factors, 10 fundamental factors (incl. momentum slope), 0 sentiment prob features, window=64
04:19:31 INFO     sentiment.features.dataloader  Built dataset: 126 windows, 16 tech factors, 10 fundamental factors (incl. momentum slope), 0 sentiment prob features, window=64
04:19:31 INFO     sentiment.features.dataloader  Built dataset: 126 windows, 16 tech factors, 10 fundamental factors (incl. momentum slope), 0 sentiment prob features, window=64


...

  Best epoch 1, val AUC=0.5362 | test windows=86

  3m: 20201001-20201231
  AAPL: 44 windows
  MSFT: 44 windows


04:23:03 INFO     sentiment.features.dataloader  Built dataset: 44 windows, 16 tech factors, 10 fundamental factors (incl. momentum slope), 0 sentiment prob features, window=64
04:23:03 INFO     sentiment.features.dataloader  Built dataset: 44 windows, 16 tech factors, 10 fundamental factors (incl. momentum slope), 0 sentiment prob features, window=64
04:23:03 INFO     sentiment.features.dataloader  Built dataset: 44 windows, 16 tech factors, 10 fundamental factors (incl. momentum slope), 0 sentiment prob features, window=64
04:23:03 INFO     sentiment.features.dataloader  Built dataset: 44 windows, 16 tech factors, 10 fundamental factors (incl. momentum slope), 0 sentiment prob features, window=64
04:23:03 INFO     sentiment.features.dataloader  Built dataset: 44 windows, 16 tech factors, 10 fundamental factors (incl. momentum slope), 0 sentiment prob features, window=64
04:23:03 INFO     sentiment.features.dataloader  Built dataset: 44 windows, 16 tech factors, 10 fundamental factors

  NVDA: 44 windows
  AMZN: 44 windows
  GOOGL: 44 windows
  META: 44 windows
  TSLA: 44 windows
  AVGO: 44 windows
  PEP: 44 windows
  COST: 44 windows


04:23:03 INFO     sentiment.model.train  Epoch   1 | train_loss=0.7301 | val_loss=0.6823 | val_auc=0.6201 | val_acc=0.6111
04:23:03 INFO     sentiment.model.train  Epoch   2 | train_loss=0.6996 | val_loss=0.6824 | val_auc=0.6136 | val_acc=0.6111
04:23:03 INFO     sentiment.model.train  Epoch   3 | train_loss=0.6843 | val_loss=0.6823 | val_auc=0.6104 | val_acc=0.6667
04:23:04 INFO     sentiment.model.train  Epoch   4 | train_loss=0.6528 | val_loss=0.6773 | val_auc=0.6201 | val_acc=0.6667
04:23:04 INFO     sentiment.model.train  Epoch   5 | train_loss=0.6529 | val_loss=0.6713 | val_auc=0.5974 | val_acc=0.6667
04:23:04 INFO     sentiment.model.train  Epoch   6 | train_loss=0.6341 | val_loss=0.6790 | val_auc=0.6039 | val_acc=0.6667
04:23:04 INFO     sentiment.model.train  Epoch   7 | train_loss=0.6352 | val_loss=0.6962 | val_auc=0.6169 | val_acc=0.6111
04:23:05 INFO     sentiment.model.train  Epoch   8 | train_loss=0.6184 | val_loss=0.7007 | val_auc=0.6201 | val_acc=0.6111
04:23:05 INFO   

  Best epoch 1, val AUC=0.6201 | test windows=88

Case study loop done.


## Plot Function

In [20]:
def plot_period_case(symbol, df_period, start, end, plots_dir):
    """Save a 2-subplot PNG for one (symbol, period)."""
    if df_period.empty:
        print(f"  {symbol} {start.date()}–{end.date()}: no data, skipping.")
        return

    start_str = start.strftime("%Y-%m-%d")
    end_str   = end.strftime("%Y-%m-%d")
    title     = f"{symbol}_{start_str}_{end_str}"

    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8), sharex=True)
    fig.suptitle(title, fontsize=13, fontweight="bold")

    # --- (a) actual close price + predicted probability --------
    color_price = "steelblue"
    color_prob  = "darkorange"

    ax1.plot(df_period["date"], df_period["close"],
             color=color_price, linewidth=1.5, label="Close Price")
    ax1.set_ylabel("Close Price (USD)", color=color_price)
    ax1.tick_params(axis="y", labelcolor=color_price)

    ax1b = ax1.twinx()
    ax1b.plot(df_period["date"], df_period["pred_prob"],
              color=color_prob, linewidth=1.2, alpha=0.85, label="Pred Prob")
    ax1b.axhline(0.5, color="gray", linestyle="--", linewidth=0.8, alpha=0.6)
    ax1b.set_ylabel("Predicted Probability", color=color_prob)
    ax1b.set_ylim(0, 1)
    ax1b.tick_params(axis="y", labelcolor=color_prob)

    lines1, labels1 = ax1.get_legend_handles_labels()
    lines2, labels2 = ax1b.get_legend_handles_labels()
    ax1.legend(lines1 + lines2, labels1 + labels2, loc="upper left", fontsize=8)
    ax1.set_title("(a) Actual Close Price vs Predicted Probability", fontsize=10)

    # --- (b) correct (+1) / wrong (-1) bar plot ----------------
    bar_colors = ["#2ecc71" if c == 1 else "#e74c3c" for c in df_period["correct"]]
    ax2.bar(df_period["date"], df_period["correct"],
            color=bar_colors, width=1.0, edgecolor="none")
    ax2.axhline(0, color="black", linewidth=0.8)
    ax2.set_yticks([-1, 1])
    ax2.set_yticklabels(["Wrong (−1)", "Correct (+1)"])
    ax2.set_ylabel("Prediction Result")
    ax2.set_xlabel("Date")

    acc = (df_period["correct"] == 1).mean()
    n   = len(df_period)
    ax2.set_title(
        f"(b) Correct (+1) vs Wrong (−1)  |  Accuracy: {acc:.1%} ({int(acc*n)}/{n})",
        fontsize=10,
    )

    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()

    save_path = plots_dir / f"{title}.png"
    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"  Saved: {save_path.name}")

## Generate and Save All Plots

In [21]:
for (period_type, start_str, end_str, symbol), df_test in all_results.items():
    subfolder = PLOTS_DIR / PERIOD_SUBFOLDER[period_type]
    start_dt  = datetime.strptime(start_str, "%Y%m%d")
    end_dt    = datetime.strptime(end_str,   "%Y%m%d")
    plot_period_case(symbol, df_test, start_dt, end_dt, subfolder)

print(f"\nAll plots saved to: {PLOTS_DIR.resolve()}")

  Saved: AAPL_2018-01-01_2018-12-31.png
  Saved: MSFT_2018-01-01_2018-12-31.png
  Saved: NVDA_2018-01-01_2018-12-31.png
  Saved: AMZN_2018-01-01_2018-12-31.png
  Saved: GOOGL_2018-01-01_2018-12-31.png
  Saved: META_2018-01-01_2018-12-31.png
  Saved: TSLA_2018-01-01_2018-12-31.png
  Saved: AVGO_2018-01-01_2018-12-31.png
  Saved: PEP_2018-01-01_2018-12-31.png
  Saved: COST_2018-01-01_2018-12-31.png
  Saved: AAPL_2018-07-01_2018-12-31.png
  Saved: MSFT_2018-07-01_2018-12-31.png
  Saved: NVDA_2018-07-01_2018-12-31.png
  Saved: AMZN_2018-07-01_2018-12-31.png
  Saved: GOOGL_2018-07-01_2018-12-31.png
  Saved: META_2018-07-01_2018-12-31.png
  Saved: TSLA_2018-07-01_2018-12-31.png
  Saved: AVGO_2018-07-01_2018-12-31.png
  Saved: PEP_2018-07-01_2018-12-31.png
  Saved: COST_2018-07-01_2018-12-31.png
  Saved: AAPL_2018-07-01_2018-09-30.png
  Saved: MSFT_2018-07-01_2018-09-30.png
  Saved: NVDA_2018-07-01_2018-09-30.png
  Saved: AMZN_2018-07-01_2018-09-30.png
  Saved: GOOGL_2018-07-01_2018-09-30.png

  # Save model weights, scalers, all_results DataFrames
 

In [22]:
import pickle

torch.save(model.state_dict(), "case_study_model.pt")

with open("case_study_scalers.pkl", "wb") as f:
    pickle.dump({"tech_scaler": tech_scaler, "fund_scaler": fund_scaler}, f)

with open("case_study_results.pkl", "wb") as f:
    pickle.dump(all_results, f)

print("Saved: case_study_model.pt, case_study_scalers.pkl, case_study_results.pkl")

Saved: case_study_model.pt, case_study_scalers.pkl, case_study_results.pkl


# Trading strategy